Testing the data and model

Loading the data

# Crop Yield Prediction — Preprocessing Pipeline + XGBoost
Step-by-step: load merged data → split → scale numeric features → encode categorical (`Item`) → fit XGBoost inside an sklearn `Pipeline` → evaluate → inspect importances.

Uses the `merged_training_table.csv` produced by `process.py` (one row per crop-year, target = `Yield` in kg/ha).

In [2]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb


## 2. Load the merged table
This is the output of `process.py` — one row per (Year, Item), already joined with the fertilizer/soil/temperature national-yearly features. We drop the pre-built `item_*` one-hot columns here since the pipeline's `OneHotEncoder` will handle that step itself — keeping raw `Item` as a single categorical column is cleaner and avoids hardcoding categories.

In [4]:
pd.set_option('display.max_columns', 20)

DATA_PATH = Path("../output/merged_training_table.csv")

df = pd.read_csv(DATA_PATH)
item_dummy_cols = [c for c in df.columns if c.startswith("item_")]
df = df.drop(columns=item_dummy_cols)

print(df.shape)
df.head()

(5788, 199)


,Year,Item,Data_Reliability,Yield,Area harvested,Production,"fertprod__Ammonia, anhydrous__Export quantity","fertprod__Ammonia, anhydrous__Export value","fertprod__Ammonia, anhydrous__Import quantity","fertprod__Ammonia, anhydrous__Import value",...,temp__Meteorological year__Standard Deviation,temp__Meteorological year__Temperature change,temp__November__Standard Deviation,temp__November__Temperature change,temp__October__Standard Deviation,temp__October__Temperature change,temp__September__Standard Deviation,temp__September__Temperature change,temp__Sep–Oct–Nov__Standard Deviation,temp__Sep–Oct–Nov__Temperature change
0,1961,"Anise, badian, coriander, cumin, caraway, fenn...",0,500.0,90000.0,45000.0,0.0,0.0,0.0,0.0,...,0.254,-0.192,0.785,-1.002,0.543,-0.701,0.29,-0.324,0.449,-0.676
1,1961,Apples,1,4157.3,44500.0,185000.0,0.0,0.0,0.0,0.0,...,0.254,-0.192,0.785,-1.002,0.543,-0.701,0.29,-0.324,0.449,-0.676
2,1961,Apricots,1,2758.6,2900.0,8000.0,0.0,0.0,0.0,0.0,...,0.254,-0.192,0.785,-1.002,0.543,-0.701,0.29,-0.324,0.449,-0.676
3,1961,Areca nuts,0,888.9,135000.0,120000.0,0.0,0.0,0.0,0.0,...,0.254,-0.192,0.785,-1.002,0.543,-0.701,0.29,-0.324,0.449,-0.676
4,1961,Bananas,1,13678.8,165000.0,2257000.0,0.0,0.0,0.0,0.0,...,0.254,-0.192,0.785,-1.002,0.543,-0.701,0.29,-0.324,0.449,-0.676


## 3. Define feature groups
- **Numeric**: `Year`, `Data_Reliability`, `Area harvested`, and every `fertprod__` / `fertnut__` / `soil__` / `temp__` macro feature — these get **scaled**.
- **Categorical**: `Item` (the crop) — this gets **one-hot encoded**.
- **Target**: `Yield` (kg/ha). `Production` is dropped from the features — it's derived from `Yield x Area harvested`, so keeping it would leak the target.

In [5]:
target_col = "Yield"

categorical_features = ["Item"]

macro_features = [c for c in df.columns
                   if c.startswith(("fertprod__", "fertnut__", "soil__", "temp__"))]

numeric_features = ["Year", "Data_Reliability"]
if "Area harvested" in df.columns:
    numeric_features.append("Area harvested")
numeric_features += macro_features

feature_cols = numeric_features + categorical_features

print(f"{len(numeric_features)} numeric features, {len(categorical_features)} categorical feature(s)")

196 numeric features, 1 categorical feature(s)


## 4. Drop rows with a missing target
Never fill a missing `Yield` with 0 — that fabricates a fake observation. Rows genuinely missing the label are dropped, not imputed.

In [6]:
df = df.dropna(subset=[target_col])
X = df[feature_cols].copy()
y = df[target_col].to_numpy(dtype=np.float32)
years = df["Year"].to_numpy()

print(X.shape, y.shape)

(5788, 197) (5788,)


## 5. Time-based train/test split
This is panel data (many crops x many years) being used for forecasting, so the split has to respect time — a random split would let the model train on 2024 rows for a crop and test on 2020 rows for the *same* crop, which overstates real accuracy. Holding out the most recent years is the honest evaluation.

In [7]:
TEST_YEARS_FRACTION = 0.15

unique_years = np.sort(years).astype(int)
unique_years = np.unique(unique_years)
cutoff_year = unique_years[int(len(unique_years) * (1 - TEST_YEARS_FRACTION))]

train_mask = years < cutoff_year
test_mask = years >= cutoff_year

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {years[train_mask].min()}-{years[train_mask].max()}  ({train_mask.sum()} rows)")
print(f"Test:  {years[test_mask].min()}-{years[test_mask].max()}  ({test_mask.sum()} rows)")

Train: 1961-2014  (4846 rows)
Test:  2015-2024  (942 rows)


## 6. Build the preprocessing + model pipeline
`ColumnTransformer` applies `StandardScaler` to numeric columns and `OneHotEncoder` to `Item` in one object, so the exact same transform is guaranteed at train and inference time — no risk of the notebook and production code drifting apart on how encoding was done.

`handle_unknown="ignore"` means a crop the encoder never saw in training won't crash inference — it just gets an all-zero encoding instead.


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), numeric_features),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform="pandas")

model = xgb.XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('encode', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature

## 7. Fit the pipeline
Fitting `pipeline` fits the scaler + encoder on the *training* data only, transforms both train and test through those fitted transforms, then trains XGBoost — this is the correct order to avoid leaking test-set statistics into the scaler.

In [10]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](197,)","['Year','Data_Reliability','Area harvested',..., 'temp__Sep–Oct–Nov__Standard Deviation', 'temp__Sep–Oct–Nov__Temperature change','Item']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,197
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('encode', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`Co

## 8. Evaluate on the held-out years

In [19]:
preds = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
mape = np.mean(np.abs((y_test - preds) / np.clip(y_test, 1e-6, None)), keepdims=False) * 100
r2 = r2_score(y_test, preds)

print(f"RMSE: {rmse:.1f} kg/ha")
print(f"MAE:  {mae:.1f} kg/ha")
print(f"MAPE: {mape:.1f}%")
print(f"R2:   {r2:.3f}")

RMSE: 3906.6 kg/ha
MAE:  2489.4 kg/ha
MAPE: 78.6%
R2:   0.916


## 9. Per-crop accuracy — for a real fluctuation range
The pooled MAPE above is misleading: sugarcane-scale crops and low-yield crops share one number even though their errors are nowhere near comparable. Instead, compute each crop's **own** MAPE/RMSE on the held-out years, and use that crop's own error rate to build a prediction range later — 'roughly X ± crop-specific %' rather than one blanket ±79%.

In [15]:
test_items = X_test["Item"].to_numpy()

crop_errors = pd.DataFrame({
    "Item": test_items,
    "y_true": y_test,
    "y_pred": preds,
})
crop_errors["abs_pct_error"] = (
    np.abs(crop_errors["y_true"] - crop_errors["y_pred"])
    / np.clip(crop_errors["y_true"], 1e-6, None) * 100
)

crop_error_stats = (
    crop_errors.groupby("Item")
    .agg(
        n_test_rows=("y_true", "size"),
        mape=("abs_pct_error", "mean"),
        rmse=("y_true", lambda s: np.sqrt(np.mean((s - crop_errors.loc[s.index, "y_pred"]) ** 2))),
    )
    .sort_values("mape")
)

crop_error_stats["low_confidence"] = crop_error_stats["n_test_rows"] < 3

overall_fallback_mape = crop_errors["abs_pct_error"].mean()

print(f"Crops covered in test set: {len(crop_error_stats)} / {df['Item'].nunique()}")
print(f"Overall fallback MAPE (used for crops absent from the test years): {overall_fallback_mape:.1f}%")
crop_error_stats.head(10)

Crops covered in test set: 95 / 96
Overall fallback MAPE (used for crops absent from the test years): 78.6%


,n_test_rows,mape,rmse,low_confidence
Item,,,,
Lettuce and chicory,10,2.547827,164.769531,False
Other stone fruits,10,2.798353,132.910553,False
Pomelos and grapefruits,2,6.210650,1542.086914,True
Bananas,10,6.515779,2798.465088,False
Papayas,10,8.103377,3876.286865,False
Eggplants (aubergines),10,8.364870,1765.874756,False
Plums and sloes,10,8.939277,339.373169,False
"Other fruits, n.e.c.",10,9.063318,1035.433472,False
Pears,10,9.736155,872.822144,False


## 10. Feature importance
Pull the feature names back out of the fitted `ColumnTransformer` (scaled numeric names unchanged, `Item` expanded to one column per crop) and pair them with XGBoost's importances.

In [16]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pd.Series(
    pipeline.named_steps["model"].feature_importances_,
    index=feature_names,
).sort_values(ascending=False)

importances.head(20)

Item_Sugar Crops Primary                                0.235321
Item_Sugar cane                                         0.231926
Item_Bananas                                            0.036310
Item_Papayas                                            0.033845
Item_Cassava, fresh                                     0.027038
Item_Grapes                                             0.021532
Item_Cantaloupes and other melons                       0.019786
Item_Roots and Tubers, Total                            0.016990
Item_Potatoes                                           0.016777
Item_Cabbages                                           0.016447
Item_Pomelos and grapefruits                            0.012515
Item_Pigeon peas, dry                                   0.011744
Item_Tomatoes                                           0.011691
Item_Cauliflowers and broccoli                          0.011360
Item_Watermelons                                        0.010217
Item_Pineapples          

## 11. Save the fitted pipeline and per-crop error stats
Saving the whole `Pipeline` (not just the model) means the scaler + encoder ship together with the model — loading this one file later is enough to go from raw `Item`/`Year`/etc. straight to a prediction, no separate preprocessing code needed.

In [17]:
OUT_DIR = Path("../output")
OUT_DIR.mkdir(exist_ok=True)

joblib.dump(pipeline, OUT_DIR / "yield_pipeline.joblib")
crop_error_stats.to_csv(OUT_DIR / "crop_error_stats.csv")

print(f"Saved pipeline to {OUT_DIR / 'yield_pipeline.joblib'}")
print(f"Saved per-crop error stats to {OUT_DIR / 'crop_error_stats.csv'}")

Saved pipeline to ../output/yield_pipeline.joblib
Saved per-crop error stats to ../output/crop_error_stats.csv
